## 1️⃣ Setup Environment

In [ ]:
# @title 1.1 🖥️ Check GPU & Select Mode
# @markdown Tự động detect GPU và chọn mode phù hợp

import subprocess
import torch

def check_gpu_and_select_mode():
    """Check GPU type and select appropriate mode"""
    if not torch.cuda.is_available():
        print("❌ No GPU detected - Using DEMO MODE")
        return "demo", 0
    
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'], 
            capture_output=True, text=True
        )
        gpu_info = result.stdout.strip()
        print(f"🖥️ GPU Info: {gpu_info}")
        
        # Get VRAM in GB
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        
        if vram_gb >= 35:
            print(f"\n✅ High VRAM ({vram_gb:.0f}GB) - FULL MODE available")
            return "full", vram_gb
        elif vram_gb >= 14:
            print(f"\n⚠️ Medium VRAM ({vram_gb:.0f}GB) - LITE MODE")
            print("   Full pipeline needs 20GB+, using BiomedCLIP + DINO only")
            return "lite", vram_gb
        else:
            print(f"\n⚠️ Low VRAM ({vram_gb:.0f}GB) - DEMO MODE")
            return "demo", vram_gb
    except Exception as e:
        print(f"❌ Error checking GPU: {e}")
        return "demo", 0

MODE, VRAM_GB = check_gpu_and_select_mode()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\n{'='*50}")
print(f"📊 Selected Mode: {MODE.upper()}")
print(f"📊 Device: {DEVICE}")
print(f"📊 Available VRAM: {VRAM_GB:.1f}GB")
print(f"{'='*50}")

In [ ]:
# @title 1.2 📦 Clone Repository & Install Dependencies
# @markdown Cài đặt packages cần thiết

import os

# ========== CONFIG ==========
REPO_URL = "https://github.com/YOUR_USERNAME/TriMedAgent.git"  # 👈 Thay bằng repo của bạn
# ============================

# Clone repo
if not os.path.exists('TriMedAgent'):
    print("📥 Cloning repository...")
    !git clone {REPO_URL}
    
%cd TriMedAgent

# Install base dependencies
print("\n📦 Installing dependencies...")
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers==4.36.0 accelerate==0.25.0
!pip install -q open_clip_torch==2.23.0
!pip install -q einops timm safetensors sentencepiece pillow requests gradio

# Mode-specific installations
if MODE in ["full", "lite"]:
    print("\n📦 Installing detection models...")
    !pip install -q groundingdino-py
    
if MODE == "full":
    print("\n📦 Installing segmentation models...")
    !pip install -q segment-anything

print("\n✅ Dependencies installed!")

In [ ]:
# @title 1.3 📚 Import Libraries

import sys
import os
import json
import numpy as np
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from io import BytesIO
import base64
import requests
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any
import time

# Add project to path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Imports successful")
print(f"   Project Root: {PROJECT_ROOT}")
print(f"   Mode: {MODE}")

In [ ]:
# @title 1.4 ⚙️ Load Configuration

CONFIG_PATH = PROJECT_ROOT / "serve" / "labels.json"

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, 'r') as f:
        TRIMED_CONFIG = json.load(f)
    print("✅ Loaded config from serve/labels.json")
else:
    TRIMED_CONFIG = {
        "triage_labels": [
            "Chest X-ray", "Brain MRI", "Abdominal CT", "Histopathology",
            "Ultrasound", "Dermoscopy", "Gross pathology", "Bone X-ray",
            "Lung CT", "Retinal fundus", "Mammography"
        ],
        "gatekeeper_prompts": {
            "positive": "Pathological finding, lesion, tumor, abnormality",
            "negative": "Normal tissue, healthy anatomy, background noise"
        },
        "thresholds": {
            "triage_confidence": 0.5,
            "gatekeeper_confidence": 0.6
        }
    }
    print("⚠️ Using default configuration")

print(f"\n📋 Triage Labels: {len(TRIMED_CONFIG['triage_labels'])} modalities")
print(f"📋 Gatekeeper Threshold: {TRIMED_CONFIG['thresholds']['gatekeeper_confidence']}")

---
## 2️⃣ 🔧 Individual Tools Demo

Demo từng tool riêng biệt (không cần Orchestrator).

In [ ]:
# @title 2.1 🔬 BiomedCLIP Tool (Triage + Gatekeeper)
# @markdown Zero-shot medical image classification

import torch
import open_clip

class BiomedCLIPTool:
    """BiomedCLIP for medical image triage and verification."""
    
    def __init__(self, device="cuda"):
        self.device = device
        self.model = None
        self.preprocess = None
        self.tokenizer = None
        
    def load(self):
        """Load BiomedCLIP model."""
        print("📥 Loading BiomedCLIP...")
        
        self.model, self.preprocess, _ = open_clip.create_model_and_transforms(
            'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224',
            device=self.device
        )
        self.tokenizer = open_clip.get_tokenizer(
            'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
        )
        self.model.eval()
        print(f"✅ BiomedCLIP loaded on {self.device}")
        return self
    
    @torch.no_grad()
    def classify(self, image: Image.Image, labels: List[str]) -> Dict[str, float]:
        """Zero-shot classification."""
        if self.model is None:
            self.load()
            
        # Preprocess image
        image_input = self.preprocess(image).unsqueeze(0).to(self.device)
        
        # Tokenize labels
        text_inputs = self.tokenizer(labels).to(self.device)
        
        # Get features
        image_features = self.model.encode_image(image_input)
        text_features = self.model.encode_text(text_inputs)
        
        # Normalize
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        
        # Calculate similarity
        similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        probs = similarity[0].cpu().numpy()
        
        return {label: float(prob) for label, prob in zip(labels, probs)}
    
    def triage(self, image: Image.Image) -> tuple:
        """Determine medical image modality."""
        labels = TRIMED_CONFIG["triage_labels"]
        scores = self.classify(image, labels)
        
        best_label = max(scores, key=scores.get)
        best_score = scores[best_label]
        
        return best_label, best_score, scores
    
    def verify_region(self, image: Image.Image, target_label: str) -> tuple:
        """Verify if a region contains pathology (Gatekeeper)."""
        pos_label = f"{TRIMED_CONFIG['gatekeeper_prompts']['positive']} of {target_label}"
        neg_label = TRIMED_CONFIG['gatekeeper_prompts']['negative']
        
        scores = self.classify(image, [pos_label, neg_label])
        
        pathology_score = scores[pos_label]
        normal_score = scores[neg_label]
        threshold = TRIMED_CONFIG['thresholds']['gatekeeper_confidence']
        
        is_valid = pathology_score > threshold
        
        return is_valid, pathology_score, normal_score

# Initialize (don't load yet)
biomedclip_tool = None

if MODE != "demo":
    biomedclip_tool = BiomedCLIPTool(device=DEVICE)
    print("🔬 BiomedCLIP tool ready (call .load() to initialize)")
else:
    print("⚠️ DEMO MODE: BiomedCLIP simulated")

In [ ]:
# @title 2.2 🎯 Grounding DINO Tool (Object Detection)
# @markdown Text-guided object detection

class GroundingDINOTool:
    """Grounding DINO for text-guided detection."""
    
    def __init__(self, device="cuda"):
        self.device = device
        self.model = None
        
    def load(self):
        """Load Grounding DINO model."""
        print("📥 Loading Grounding DINO...")
        
        try:
            from groundingdino.util.inference import load_model, predict
            import groundingdino.datasets.transforms as T
            
            # Load model (adjust path as needed)
            self.model = load_model(
                "groundingdino/config/GroundingDINO_SwinT_OGC.py",
                "weights/groundingdino_swint_ogc.pth"
            )
            self.predict_fn = predict
            self.transform = T.Compose([
                T.RandomResize([800], max_size=1333),
                T.ToTensor(),
                T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
            ])
            print(f"✅ Grounding DINO loaded on {self.device}")
            
        except Exception as e:
            print(f"⚠️ Could not load Grounding DINO: {e}")
            print("   Using mock detection instead")
            self.model = "mock"
            
        return self
    
    def detect(self, image: Image.Image, query: str, 
               box_threshold: float = 0.25, 
               text_threshold: float = 0.25) -> Dict:
        """Detect objects based on text query."""
        
        if self.model == "mock" or self.model is None:
            # Mock detection for demo
            w, h = image.size
            return {
                "boxes": [[0.2*w, 0.3*h, 0.6*w, 0.7*h]],
                "phrases": [query],
                "logits": [0.75]
            }
        
        # Real detection
        image_tensor, _ = self.transform(image, None)
        boxes, logits, phrases = self.predict_fn(
            model=self.model,
            image=image_tensor,
            caption=query,
            box_threshold=box_threshold,
            text_threshold=text_threshold
        )
        
        # Convert to pixel coordinates
        w, h = image.size
        boxes_pixel = []
        for box in boxes:
            cx, cy, bw, bh = box
            x1 = (cx - bw/2) * w
            y1 = (cy - bh/2) * h
            x2 = (cx + bw/2) * w
            y2 = (cy + bh/2) * h
            boxes_pixel.append([x1, y1, x2, y2])
        
        return {
            "boxes": boxes_pixel,
            "phrases": phrases,
            "logits": logits.tolist()
        }

# Initialize
dino_tool = None
if MODE in ["full", "lite"]:
    dino_tool = GroundingDINOTool(device=DEVICE)
    print("🎯 Grounding DINO tool ready")
else:
    print("⚠️ DEMO MODE: DINO simulated")

In [ ]:
# @title 2.3 🎭 MedSAM Tool (Segmentation)
# @markdown Medical image segmentation

class MedSAMTool:
    """MedSAM for medical image segmentation."""
    
    def __init__(self, device="cuda"):
        self.device = device
        self.model = None
        
    def load(self):
        """Load MedSAM model."""
        print("📥 Loading MedSAM...")
        
        try:
            from segment_anything import sam_model_registry, SamPredictor
            
            # Load model
            sam = sam_model_registry["vit_b"](checkpoint="weights/medsam_vit_b.pth")
            sam.to(self.device)
            self.predictor = SamPredictor(sam)
            print(f"✅ MedSAM loaded on {self.device}")
            
        except Exception as e:
            print(f"⚠️ Could not load MedSAM: {e}")
            print("   Using mock segmentation instead")
            self.predictor = "mock"
            
        return self
    
    def segment(self, image: Image.Image, boxes: List[List[float]]) -> List[np.ndarray]:
        """Segment regions defined by boxes."""
        
        if self.predictor == "mock" or self.predictor is None:
            # Mock segmentation for demo
            masks = []
            img_array = np.array(image)
            h, w = img_array.shape[:2]
            
            for box in boxes:
                x1, y1, x2, y2 = [int(c) for c in box]
                mask = np.zeros((h, w), dtype=np.uint8)
                # Create ellipse mask inside box
                cy, cx = (y1 + y2) // 2, (x1 + x2) // 2
                ry, rx = (y2 - y1) // 2, (x2 - x1) // 2
                for y in range(max(0, y1), min(h, y2)):
                    for x in range(max(0, x1), min(w, x2)):
                        if ((x - cx) / rx) ** 2 + ((y - cy) / ry) ** 2 <= 1:
                            mask[y, x] = 255
                masks.append(mask)
            
            return masks
        
        # Real segmentation
        img_array = np.array(image)
        self.predictor.set_image(img_array)
        
        masks = []
        for box in boxes:
            mask, _, _ = self.predictor.predict(
                box=np.array(box),
                multimask_output=False
            )
            masks.append((mask[0] * 255).astype(np.uint8))
        
        return masks

# Initialize
medsam_tool = None
if MODE == "full":
    medsam_tool = MedSAMTool(device=DEVICE)
    print("🎭 MedSAM tool ready")
else:
    print(f"⚠️ {MODE.upper()} MODE: MedSAM {'simulated' if MODE == 'demo' else 'not available'}")

In [ ]:
# @title 2.4 🧪 Test Individual Tools
# @markdown Chạy test với ảnh mẫu

# Download sample image
SAMPLE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/c/c4/Chest_Xray_PA_3-8-2010.png/220px-Chest_Xray_PA_3-8-2010.png"

print("📥 Downloading sample image...")
response = requests.get(SAMPLE_URL)
sample_image = Image.open(BytesIO(response.content)).convert("RGB")
print(f"✅ Sample image: {sample_image.size}")

# Display
plt.figure(figsize=(6, 6))
plt.imshow(sample_image)
plt.title("Sample Medical Image")
plt.axis('off')
plt.show()

In [ ]:
# @title 2.4.1 Test BiomedCLIP (Triage)

if biomedclip_tool is not None:
    print("🔬 Loading BiomedCLIP...")
    biomedclip_tool.load()
    
    print("\n🔬 Running Triage...")
    modality, confidence, all_scores = biomedclip_tool.triage(sample_image)
    
    print(f"\n📊 Triage Result:")
    print(f"   Modality: {modality}")
    print(f"   Confidence: {confidence:.1%}")
    
    print(f"\n   Top 5 Scores:")
    sorted_scores = sorted(all_scores.items(), key=lambda x: x[1], reverse=True)
    for label, score in sorted_scores[:5]:
        bar = "█" * int(score * 30)
        print(f"   {label}: {score:.1%} {bar}")
else:
    # Demo mode
    print("🔬 [DEMO] Triage Result:")
    print("   Modality: Chest X-ray")
    print("   Confidence: 92.5%")

In [ ]:
# @title 2.4.2 Test Grounding DINO (Detection)

if dino_tool is not None:
    print("🎯 Loading Grounding DINO...")
    dino_tool.load()
    
    print("\n🎯 Running Detection...")
    detection_query = "lung nodule"
    detection_result = dino_tool.detect(sample_image, detection_query)
    
    print(f"\n📊 Detection Result:")
    print(f"   Query: '{detection_query}'")
    print(f"   Boxes Found: {len(detection_result['boxes'])}")
    
    # Visualize
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(sample_image)
    
    for i, (box, phrase, logit) in enumerate(zip(
        detection_result['boxes'], 
        detection_result['phrases'],
        detection_result['logits']
    )):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor='red', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1, y1-5, f"{phrase}: {logit:.2f}", color='red', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    
    ax.set_title(f"Detection: '{detection_query}'")
    ax.axis('off')
    plt.show()
else:
    print("🎯 [DEMO] Detection Result:")
    print("   Query: 'lung nodule'")
    print("   Boxes Found: 1")
    print("   Box: [88, 132, 264, 308], Score: 0.75")

---
## 3️⃣ 🎯 Orchestrator Demo

Demo **Thin Client Orchestrator** - điều phối pipeline tự động.

In [ ]:
# @title 3.1 📦 Local Orchestrator (Colab Edition)
# @markdown Orchestrator chạy trực tiếp (không cần HTTP workers)

@dataclass
class TriageResult:
    modality: str
    confidence: float
    all_scores: Dict[str, float] = field(default_factory=dict)

@dataclass
class DetectionResult:
    boxes: List[List[float]]
    labels: List[str]
    scores: List[float]

@dataclass
class GatekeeperResult:
    box: List[float]
    is_valid: bool
    pathology_score: float
    normal_score: float
    reason: str

@dataclass  
class SegmentationResult:
    masks: List[np.ndarray]

@dataclass
class PipelineResult:
    # Stage results
    triage: Optional[TriageResult] = None
    llava_response: str = ""
    context_injected: str = ""
    dino_raw_boxes: List[List[float]] = field(default_factory=list)
    dino_labels: List[str] = field(default_factory=list)
    dino_scores: List[float] = field(default_factory=list)
    gatekeeper_results: List[GatekeeperResult] = field(default_factory=list)
    verified_boxes: List[List[float]] = field(default_factory=list)
    rejected_boxes: List[List[float]] = field(default_factory=list)
    masks: List[np.ndarray] = field(default_factory=list)
    
    # Metadata
    execution_time: float = 0.0
    pipeline_complete: bool = False
    stopped_at_stage: str = ""
    errors: List[str] = field(default_factory=list)


class TriMedOrchestratorLocal:
    """
    Local Orchestrator for Colab/Kaggle.
    Runs tools directly instead of HTTP calls.
    """
    
    def __init__(self, 
                 biomedclip=None, 
                 dino=None, 
                 medsam=None,
                 config=None):
        self.biomedclip = biomedclip
        self.dino = dino
        self.medsam = medsam
        self.config = config or TRIMED_CONFIG
        
        print("🎯 Local Orchestrator initialized")
        print(f"   BiomedCLIP: {'✅' if biomedclip else '❌'}")
        print(f"   DINO: {'✅' if dino else '❌'}")
        print(f"   MedSAM: {'✅' if medsam else '❌'}")
    
    def triage(self, image: Image.Image) -> TriageResult:
        """Stage 1: Triage - Identify image modality."""
        if self.biomedclip is None:
            return TriageResult(
                modality="Chest X-ray",
                confidence=0.92,
                all_scores={"Chest X-ray": 0.92, "Brain MRI": 0.05}
            )
        
        modality, conf, scores = self.biomedclip.triage(image)
        return TriageResult(modality=modality, confidence=conf, all_scores=scores)
    
    def detect(self, image: Image.Image, query: str) -> DetectionResult:
        """Stage 3: Detection - Find objects."""
        if self.dino is None:
            w, h = image.size
            return DetectionResult(
                boxes=[[0.2*w, 0.3*h, 0.6*w, 0.7*h]],
                labels=[query],
                scores=[0.75]
            )
        
        result = self.dino.detect(image, query)
        return DetectionResult(
            boxes=result['boxes'],
            labels=result['phrases'],
            scores=result['logits']
        )
    
    def verify_box(self, image: Image.Image, box: List[float], 
                   target: str) -> GatekeeperResult:
        """Stage 4: Gatekeeper - Verify detection."""
        # Crop image to box
        x1, y1, x2, y2 = [int(c) for c in box]
        cropped = image.crop((x1, y1, x2, y2))
        
        if self.biomedclip is None:
            return GatekeeperResult(
                box=box,
                is_valid=True,
                pathology_score=0.85,
                normal_score=0.15,
                reason="[DEMO] Verified"
            )
        
        is_valid, path_score, norm_score = self.biomedclip.verify_region(cropped, target)
        
        return GatekeeperResult(
            box=box,
            is_valid=is_valid,
            pathology_score=path_score,
            normal_score=norm_score,
            reason=f"Pathology: {path_score:.1%} vs Normal: {norm_score:.1%}"
        )
    
    def segment(self, image: Image.Image, boxes: List[List[float]]) -> SegmentationResult:
        """Stage 5: Segmentation."""
        if self.medsam is None:
            # Create mock masks
            masks = []
            h, w = np.array(image).shape[:2]
            for box in boxes:
                mask = np.zeros((h, w), dtype=np.uint8)
                x1, y1, x2, y2 = [int(c) for c in box]
                mask[y1:y2, x1:x2] = 255
                masks.append(mask)
            return SegmentationResult(masks=masks)
        
        masks = self.medsam.segment(image, boxes)
        return SegmentationResult(masks=masks)
    
    def run_full_chain(self, image: Image.Image, user_query: str,
                       skip_gatekeeper: bool = False,
                       skip_segmentation: bool = False) -> PipelineResult:
        """
        Run the complete pipeline:
        Perceive → Reason → Act → Verify → Segment
        """
        start_time = time.time()
        result = PipelineResult()
        
        try:
            # ===== Stage 1: TRIAGE =====
            print("🔬 Stage 1: Triage...")
            result.triage = self.triage(image)
            print(f"   → {result.triage.modality} ({result.triage.confidence:.1%})")
            
            # Build context
            result.context_injected = f"[System: This is a {result.triage.modality} image with {result.triage.confidence:.0%} confidence]"
            
            # ===== Stage 2: REASON (Mock LLaVA) =====
            print("🤖 Stage 2: Reasoning...")
            result.llava_response = f"Based on the {result.triage.modality} image, I observe potential abnormalities that require further analysis with detection tools."
            print(f"   → Response generated")
            
            # Check if detection needed
            action_keywords = ["find", "detect", "segment", "locate", "identify", "show"]
            needs_detection = any(kw in user_query.lower() for kw in action_keywords)
            
            if not needs_detection:
                result.stopped_at_stage = "reason"
                result.pipeline_complete = True
                return result
            
            # ===== Stage 3: DETECT =====
            print("🎯 Stage 3: Detection...")
            # Extract target from query
            target = user_query.split()[-1] if user_query else "abnormality"
            
            detection = self.detect(image, target)
            result.dino_raw_boxes = detection.boxes
            result.dino_labels = detection.labels
            result.dino_scores = detection.scores
            print(f"   → Found {len(detection.boxes)} boxes")
            
            if not detection.boxes:
                result.stopped_at_stage = "detect"
                result.pipeline_complete = True
                return result
            
            # ===== Stage 4: VERIFY (Gatekeeper) =====
            if not skip_gatekeeper:
                print("✅ Stage 4: Gatekeeper Verification...")
                
                for box in detection.boxes:
                    gk_result = self.verify_box(image, box, target)
                    result.gatekeeper_results.append(gk_result)
                    
                    if gk_result.is_valid:
                        result.verified_boxes.append(box)
                    else:
                        result.rejected_boxes.append(box)
                
                print(f"   → Verified: {len(result.verified_boxes)}, Rejected: {len(result.rejected_boxes)}")
            else:
                result.verified_boxes = detection.boxes
            
            # ===== Stage 5: SEGMENT =====
            if not skip_segmentation and result.verified_boxes:
                print("🎭 Stage 5: Segmentation...")
                seg_result = self.segment(image, result.verified_boxes)
                result.masks = seg_result.masks
                print(f"   → Generated {len(result.masks)} masks")
            
            result.stopped_at_stage = "complete"
            result.pipeline_complete = True
            
        except Exception as e:
            result.errors.append(str(e))
            print(f"❌ Error: {e}")
        
        result.execution_time = time.time() - start_time
        return result

print("✅ Orchestrator class defined")

In [ ]:
# @title 3.2 🚀 Initialize Orchestrator

# Load tools if available
if biomedclip_tool and not biomedclip_tool.model:
    biomedclip_tool.load()
    
if dino_tool and not dino_tool.model:
    dino_tool.load()

if medsam_tool and not medsam_tool.model:
    medsam_tool.load()

# Create orchestrator
orchestrator = TriMedOrchestratorLocal(
    biomedclip=biomedclip_tool,
    dino=dino_tool,
    medsam=medsam_tool,
    config=TRIMED_CONFIG
)

print("\n🎯 Orchestrator ready!")

In [ ]:
# @title 3.3 ▶️ Run Full Pipeline

# Define query
user_query = "Find and segment any abnormalities in this image"

print(f"🎯 Query: '{user_query}'")
print("=" * 60)

# Run pipeline
result = orchestrator.run_full_chain(
    image=sample_image,
    user_query=user_query,
    skip_gatekeeper=False,
    skip_segmentation=(MODE != "full")
)

print("\n" + "=" * 60)
print("📋 PIPELINE SUMMARY")
print("=" * 60)
print(f"   Triage: {result.triage.modality} ({result.triage.confidence:.1%})")
print(f"   Raw Boxes: {len(result.dino_raw_boxes)}")
print(f"   Verified Boxes: {len(result.verified_boxes)}")
print(f"   Masks: {len(result.masks)}")
print(f"   Time: {result.execution_time:.2f}s")
print(f"   Complete: {result.pipeline_complete}")

In [ ]:
# @title 3.4 📊 Visualize Pipeline Results

def visualize_results(image, result):
    """Visualize pipeline results."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    img_array = np.array(image)
    
    # Panel 1: Original with Triage
    axes[0].imshow(img_array)
    axes[0].set_title(f"Input: {result.triage.modality}\n({result.triage.confidence:.1%})", fontsize=12)
    axes[0].axis('off')
    
    # Panel 2: Detection
    axes[1].imshow(img_array)
    for box in result.dino_raw_boxes:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor='red', facecolor='none', linestyle='--'
        )
        axes[1].add_patch(rect)
    
    for box in result.verified_boxes:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=3, edgecolor='lime', facecolor='none'
        )
        axes[1].add_patch(rect)
    
    axes[1].set_title(f"Detection\nRaw: {len(result.dino_raw_boxes)}, Verified: {len(result.verified_boxes)}", fontsize=12)
    axes[1].axis('off')
    
    # Panel 3: Segmentation
    axes[2].imshow(img_array)
    
    if result.masks:
        mask_overlay = np.zeros((*img_array.shape[:2], 4))
        for mask in result.masks:
            mask_bool = mask > 0
            mask_overlay[mask_bool, 0] = 0.2  # R
            mask_overlay[mask_bool, 1] = 0.6  # G  
            mask_overlay[mask_bool, 2] = 1.0  # B
            mask_overlay[mask_bool, 3] = 0.5  # A
        axes[2].imshow(mask_overlay)
    
    axes[2].set_title(f"Segmentation\nMasks: {len(result.masks)}", fontsize=12)
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_results(sample_image, result)

---
## 4️⃣ 💬 Interactive Chatbot Demo

Chatbot interface với Gradio.

In [ ]:
# @title 4.1 🤖 Medical Chatbot Interface

import gradio as gr

# Global state
current_image = None
current_result = None
chat_history = []

def process_image(image):
    """Process uploaded image."""
    global current_image
    current_image = image
    
    if image is None:
        return "⚠️ Please upload an image first."
    
    # Run triage
    triage = orchestrator.triage(image)
    
    return f"""✅ **Image Loaded**
    
📊 **Triage Result:**
- **Modality**: {triage.modality}
- **Confidence**: {triage.confidence:.1%}

💡 Try asking:
- "What can you see in this image?"
- "Find any tumors or abnormalities"
- "Describe the pathological findings"
"""

def chat(message, history):
    """Chat with the medical AI."""
    global current_image, current_result
    
    if current_image is None:
        return "⚠️ Please upload a medical image first."
    
    # Check if action requested
    action_keywords = ["find", "detect", "segment", "locate"]
    needs_action = any(kw in message.lower() for kw in action_keywords)
    
    if needs_action:
        # Run full pipeline
        current_result = orchestrator.run_full_chain(
            image=current_image,
            user_query=message
        )
        
        response = f"""🔍 **Analysis Complete**

**Triage**: {current_result.triage.modality} ({current_result.triage.confidence:.1%})

**Detection Results**:
- Raw detections: {len(current_result.dino_raw_boxes)}
- Verified boxes: {len(current_result.verified_boxes)}

**LLaVA Response**:
{current_result.llava_response}

⏱️ Processing time: {current_result.execution_time:.2f}s
"""
    else:
        # Simple Q&A
        triage = orchestrator.triage(current_image)
        response = f"""Based on the {triage.modality} image:

{orchestrator.run_full_chain(current_image, message).llava_response}

💡 *Tip: Use "find" or "detect" to trigger detection & segmentation.*
"""
    
    return response

def get_visualization():
    """Get current visualization."""
    global current_image, current_result
    
    if current_image is None:
        return None
    
    if current_result is None:
        return current_image
    
    # Create visualization
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(np.array(current_image))
    
    for box in current_result.verified_boxes:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=3, edgecolor='lime', facecolor='none'
        )
        ax.add_patch(rect)
    
    ax.axis('off')
    
    # Convert to image
    fig.canvas.draw()
    img_array = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    img_array = img_array.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    plt.close(fig)
    
    return Image.fromarray(img_array)

# Create Gradio Interface
print("🚀 Starting Chatbot Interface...")
print("   (Run this cell and click the link below)")

In [ ]:
# @title 4.2 ▶️ Launch Chatbot

with gr.Blocks(title="🏥 TriMedAgent Chatbot", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🏥 TriMedAgent - Medical AI Assistant
    
    Upload a medical image and ask questions about it!
    
    **Example queries:**
    - "What abnormalities can you see?"
    - "Find and segment any tumors"
    - "Describe the pathological findings"
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="📷 Upload Medical Image")
            upload_btn = gr.Button("📤 Analyze Image", variant="primary")
            status_output = gr.Markdown(label="Status")
        
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=400, label="💬 Chat")
            msg_input = gr.Textbox(
                placeholder="Ask about the image...",
                label="Your Question"
            )
            with gr.Row():
                send_btn = gr.Button("Send", variant="primary")
                clear_btn = gr.Button("Clear")
    
    with gr.Row():
        viz_output = gr.Image(label="📊 Detection Results")
        viz_btn = gr.Button("🔄 Update Visualization")
    
    # Event handlers
    upload_btn.click(process_image, inputs=[image_input], outputs=[status_output])
    msg_input.submit(chat, inputs=[msg_input, chatbot], outputs=[chatbot])
    send_btn.click(chat, inputs=[msg_input, chatbot], outputs=[chatbot])
    clear_btn.click(lambda: None, None, chatbot)
    viz_btn.click(get_visualization, outputs=[viz_output])

# Launch
demo.launch(share=True, debug=True)

---
## 📋 Summary

### ✅ What we covered:

1. **🔧 Individual Tools**:
   - BiomedCLIP: Medical image triage & gatekeeper
   - Grounding DINO: Text-guided object detection
   - MedSAM: Medical image segmentation

2. **🎯 Orchestrator**:
   - Local version for Colab/Kaggle
   - Full pipeline: Perceive → Reason → Act → Verify → Segment

3. **💬 Chatbot**:
   - Interactive Gradio interface
   - Upload image and chat

### 🚀 Next Steps:

- **Full Mode**: Use A100/V100 for complete pipeline
- **Production**: Deploy workers as HTTP services
- **Fine-tuning**: Train on domain-specific data

---

📚 **Documentation**: See `docs/` folder for detailed guides.